In [ ]:
!pip install easyocr

In [ ]:

import easyocr
import cv2
import numpy as np
from PIL import Image
import re
import gradio as gr

# -----------------------------
# OCR MODEL (English + Urdu)
# -----------------------------
reader = easyocr.Reader(['en', 'ur'], gpu=False)

# -----------------------------
# PREPROCESSING
# -----------------------------
def preprocess_image(image):
    img = np.array(image)

    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    thresh = cv2.adaptiveThreshold(
        blur, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        15, 3
    )

    return thresh

# -----------------------------
# CLEAN TEXT
# -----------------------------
def clean_text(text):
    text = re.sub(r'[^؀-ۿ\u0020-\u007EA-Za-z0-9\s.,!?]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# -----------------------------
# MAIN FUNCTION
# -----------------------------
def convert_and_save(image):
    if image is None:
        return "⚠ Please upload an image first", None

    # Step 1: Preprocess
    processed = preprocess_image(image)

    # Step 2: OCR
    result = reader.readtext(processed)

    raw_text = " ".join([text for (_, text, _) in result])

    # Step 3: Clean
    final_text = clean_text(raw_text)

    # Step 4: Save file
    file_path = "digital_text_output.txt"
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(final_text)

    # IMPORTANT: return BOTH text + file
    return final_text, file_path

# -----------------------------
# UI (PROFESSIONAL)
# -----------------------------
with gr.Blocks(theme=gr.themes.Soft()) as app:

    gr.Markdown("""
    #  HANDWRITTEN NOTES TO DIGITAL TEXT CONVERTER
    ### English + Urdu Handwriting → Digital Text
    """)

    image_input = gr.Image(type="pil", label="📤 Upload Handwritten Image")

    convert_btn = gr.Button(" Convert to Digital Text", variant="primary")

    text_output = gr.Textbox(label=" Digital Text Output", lines=10)

    # ✅ THIS IS THE REAL DOWNLOAD BUTTON
    file_output = gr.File(label="⬇ Download Digital Text File")

    # -----------------------------
    # ACTION
    # -----------------------------
    convert_btn.click(
        fn=convert_and_save,
        inputs=image_input,
        outputs=[text_output, file_output]
    )

# -----------------------------
# RUN APP
# -----------------------------
app.launch(share=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 36.0 MB/s eta 0:00:00


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

/tmp/ipykernel_928/2519718522.py:70: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f454a150057428e37e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
